# Health impact

We turn exposure probabilities into an SO2 increase per barangay-day and the expected extra respiratory
consultations, using dY = P x Y0 x (RR^(dC/10) - 1). Outputs:

1. `data/processed/health_impact.csv`: SO2 increase, extra cases, risk level and rank per barangay-day, for three scenarios
2. `results/tables/health_sensitivity.csv`: 2026 totals for every combination of RR, baseline rates and scenario
3. `results/tables/top_barangays_2026.csv`: the 10 barangays with the most extra cases in 2026
4. `results/tables/health_by_month.csv`: monthly extra cases, 2024 to 2026
5. `results/tables/top_risk_days.csv`: the 10 days with the most extra cases, 2024 to 2026

The reference emission level comes from 2024-2025 only, and nothing here is fit on 2026.

## Setup

In [1]:
import itertools
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import health

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REFERENCE = ROOT / "data" / "reference"
REPORT_LOG = ROOT / "data" / "report_log" / "report_log.csv"
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

TRAIN_END = pd.Timestamp("2025-12-31")
ACTIVITY_CAP = 3
SCENARIOS_UGM3 = [20, 40, 100]
MAIN_SCENARIO_UGM3 = health.WHO_SO2_24H_UGM3
RR_SENSITIVITY = [1.001, health.RR_PER_10_UGM3, 1.020]
BACOLOD_ADM2_PSGC = 630200000

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def show(df, name):
    """Print row count and missing values per column."""
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns")
    na = df.isna().sum()
    na = na[na > 0]
    print("  missing:", "none" if na.empty else na.to_dict())


metrics = {}

## Load data

In [2]:
exposure = pd.read_csv(PROCESSED / "exposure_scores.csv", parse_dates=["date"])
show(exposure, "exposure_scores")
assert len(exposure) == 221_400

forecasts = pd.read_csv(PROCESSED / "next_day_forecasts.csv", parse_dates=["date"])
show(forecasts, "next_day_forecasts")
print("forecast dates:", forecasts["date"].min().date(), "to", forecasts["date"].max().date())

barangays = (pd.read_csv(PROCESSED / "barangays_study_area.csv")
             .rename(columns={"adm4_psgc": "psgc_code", "adm4_en": "barangay"}))
show(barangays, "barangays_study_area")
assert len(barangays) == 216 and barangays["psgc_code"].is_unique

rates = pd.read_csv(REFERENCE / "baseline_rates_fhsis2022.csv")
show(rates, "baseline_rates_fhsis2022")
rates[["area", "Y0_per_person_per_day", "ari_under5_per_child_per_year", "use"]]

exposure_scores: 221,400 rows x 6 columns
  missing: {'plume_index': 216}
next_day_forecasts: 1,025 rows x 5 columns
  missing: {'drift_forecast_sector': 2, 'drift_forecast_prob': 2}
forecast dates: 2024-01-02 to 2026-10-22
barangays_study_area: 216 rows x 19 columns
  missing: none
baseline_rates_fhsis2022: 4 rows x 14 columns
  missing: none


,area,Y0_per_person_per_day,ari_under5_per_child_per_year,use
0,Negros Occidental (excl. Bacolod),0.000040,0.0210,main baseline
1,Bacolod City,0.000012,0.0074,reference
2,Negros Oriental,0.000014,0.0010,reference (likely underreported)
3,Philippines,0.000089,0.0524,upper bound for sensitivity


Observed SO2 comes from `volcano_daily.csv`. Report days are the event dates in the report log, as in
notebooks 01 and 02.

In [3]:
volcano = pd.read_csv(PROCESSED / "volcano_daily.csv", usecols=["date", "so2_mean", "so2_observed", "split"],
                      parse_dates=["date"])
show(volcano, "volcano_daily")

report_log = pd.read_csv(REPORT_LOG, usecols=["event_date", "psgc_code", "hazard"], parse_dates=["event_date"])
report_log["psgc_code"] = report_log["psgc_code"].astype("Int64")
show(report_log, "report_log")
report_dates = pd.DatetimeIndex(sorted(report_log["event_date"].unique()))
print(f"report days: {len(report_dates)} ({(report_dates <= TRAIN_END).sum()} in 2024-2025, "
      f"{(report_dates > TRAIN_END).sum()} in 2026)")

volcano_daily: 1,025 rows x 4 columns
  missing: {'so2_mean': 170}
report_log: 243 rows x 3 columns
  missing: {'psgc_code': 1}
report days: 14 (9 in 2024-2025, 5 in 2026)


The study-area table has no municipality name, so names come from the `Mun` column of the 2020 census, joined
by PSGC code.

In [4]:
census = pd.read_excel(RAW / "2020-census-total-popn_single-age_malefemale_admin4.xlsx", sheet_name="Brgy_adm4",
                       usecols=["BgyCode_new", "Mun"])
show(census, "census")
census["psgc_code"] = pd.to_numeric(census["BgyCode_new"].str.replace("PH", "", regex=False), errors="coerce")
municipalities = (census[census["psgc_code"].isin(barangays["psgc_code"])]
                  .astype({"psgc_code": "int64"})
                  .rename(columns={"Mun": "municipality"})[["psgc_code", "municipality"]])
show(municipalities, "census rows for study barangays")
assert len(municipalities) == 216 and municipalities["psgc_code"].is_unique

barangays = barangays.merge(municipalities, on="psgc_code", how="left", validate="1:1")
show(barangays, "barangays + municipality")

census: 42,041 rows x 2 columns
  missing: none
census rows for study barangays: 216 rows x 2 columns
  missing: none
barangays + municipality: 216 rows x 20 columns
  missing: none


## Reference emission level

The exposure probability says where a plume would be felt on an event day, not how much SO2 arrives, so we scale it
by the forecast emission relative to q_ref, the median observed SO2 on the 2024-2025 report days. We fixed q_ref
before looking at any 2026 result and keep it at that value.

In [5]:
train_report_days = volcano[volcano["date"].isin(report_dates) & (volcano["date"] <= TRAIN_END)]
print(train_report_days[["date", "so2_mean", "so2_observed"]].to_string(index=False))
q_ref_days = train_report_days[train_report_days["so2_observed"] == 1]
Q_REF_TD = q_ref_days["so2_mean"].median()
print(f"q_ref = {Q_REF_TD:,.1f} t/d, median of the {len(q_ref_days)} of {len(train_report_days)} "
      "2024-2025 report days with an SO2 reading")
train_median_td = volcano.loc[(volcano["date"] <= TRAIN_END) & (volcano["so2_observed"] == 1), "so2_mean"].median()
print(f"for comparison, median of all 2024-2025 days with a reading: {train_median_td:,.1f} t/d")

      date  so2_mean  so2_observed
2024-06-03       NaN             0
2024-09-11   11556.0             1
2024-09-24    8973.0             1
2024-10-28   10074.0             1
2024-11-15    5024.0             1
2024-12-09    1669.0             1
2025-02-15    2625.0             1
2025-04-08    4992.5             1
2025-10-24     189.0             1
q_ref = 5,008.2 t/d, median of the 8 of 9 2024-2025 report days with an SO2 reading
for comparison, median of all 2024-2025 days with a reading: 2,221.0 t/d


## Activity factor

activity = forecast SO2 / q_ref, capped at 3, so a barangay's SO2 increase is at most three times its value on a
typical 2024-2025 report day. The forecast is notebook 02's next-day SO2 forecast, made with data up to the day
before (an in-sample fit on 2024-2025 dates).

In [6]:
impact = exposure.merge(forecasts[["date", "so2_forecast_td"]], on="date", how="left", validate="m:1")
show(impact, "exposure_scores + forecasts")
no_forecast = impact["so2_forecast_td"].isna()
print("dates without a forecast:", sorted(impact.loc[no_forecast, "date"].dt.date.astype(str).unique()))
impact = impact[~no_forecast].reset_index(drop=True)
show(impact, "barangay-days with a forecast")

impact["activity"] = health.activity_factor(impact["so2_forecast_td"], Q_REF_TD, ACTIVITY_CAP)
impact["report_day"] = impact["date"].isin(report_dates).astype(int)

daily = impact.drop_duplicates("date")[["date", "split", "so2_forecast_td", "activity"]].copy()
daily["at_cap"] = daily["so2_forecast_td"] / Q_REF_TD >= ACTIVITY_CAP
activity_summary = daily.groupby("split").agg(days=("date", "size"), min=("activity", "min"),
                                              median=("activity", "median"), max=("activity", "max"),
                                              days_at_cap=("at_cap", "sum"))
print("Activity factor per day")
activity_summary.round(3)

exposure_scores + forecasts: 221,400 rows x 7 columns
  missing: {'plume_index': 216, 'so2_forecast_td': 216}
dates without a forecast: ['2024-01-01']
barangay-days with a forecast: 221,184 rows x 7 columns
  missing: {'plume_index': 216}
Activity factor per day


,days,min,median,max,days_at_cap
split,,,,,
test,294,0.065,0.329,0.495,0
train,730,0.160,0.373,1.460,0


The cap is never reached. Emission in 2026 was lower than on the 2024-2025 report days, so every 2026 forecast is
below half of q_ref.

## Exposure scenarios

delta_c = p_exposed x activity x S is the increase in 24-hour SO2 (ug/m3), where S is the increase we assume for a
barangay that is certainly exposed on a typical report day. Our SO2 data are emission rates at the crater (t/d), not
concentrations where people live, so S is a scenario: 20, 40 or 100 ug/m3, with S = 40, the WHO 24-hour guideline,
as the main scenario.

## Extra respiratory consultations

For each scenario and barangay-day, with RR = 1.0126 per 10 ug/m3 (Carlsen et al., 2021):

- extra_under5 = pop_under5 x Y0_under5 x (RR^(delta_c/10) - 1)
- extra_60plus = pop_60plus x Y0_all x (RR^(delta_c/10) - 1)
- extra_all = pop_total x Y0_all x (RR^(delta_c/10) - 1)
- pct_rise = (RR^(delta_c/10) - 1) x 100

extra_all counts every resident at the all-ages rate, so it is not the sum of the two groups.

Baseline rates come from FHSIS 2022 (`data/reference/baseline_rates_fhsis2022.csv`):

| Barangays | Y0_all (per person per day) | Y0_under5 (per child per day) |
|---|---|---|
| Bacolod City | 0.0000121 | 0.0074 / 365 |
| Negros Occidental and Negros Oriental | 0.0000397 | 0.0210 / 365 |

Bacolod City barangays (PSGC 630200000) use Bacolod City's own rates. Negros Oriental barangays use the Negros
Occidental rates, because the Negros Oriental figures look underreported (0.0010 under-5 ARI cases per child per
year, against 0.0210 in Negros Occidental). FHSIS has no rate for adults aged 60 and over, so we assume they
have the all-ages rate.

In [7]:
DAILY_RATES = {row.area: {"under5": row.ari_under5_per_child_per_year / 365, "all_ages": row.Y0_per_person_per_day}
               for row in rates.itertuples()}
OCCIDENTAL_AREA = rates.loc[rates["use"] == "main baseline", "area"].item()
BACOLOD_AREA = "Bacolod City"
PHILIPPINES_AREA = rates.loc[rates["use"] == "upper bound for sensitivity", "area"].item()
MAIN_BASELINE = "Negros Occidental and Bacolod City"
for area in [OCCIDENTAL_AREA, BACOLOD_AREA, PHILIPPINES_AREA]:
    print(f"{area}: under 5 {DAILY_RATES[area]['under5']:.4e}, "
          f"all ages {DAILY_RATES[area]['all_ages']:.4e} cases per person per day")

barangays["rate_area"] = np.where(barangays["adm2_psgc"] == BACOLOD_ADM2_PSGC, BACOLOD_AREA, OCCIDENTAL_AREA)
barangays["y0_under5_per_day"] = barangays["rate_area"].map(lambda area: DAILY_RATES[area]["under5"])
barangays["y0_all_per_day"] = barangays["rate_area"].map(lambda area: DAILY_RATES[area]["all_ages"])
print(barangays.groupby(["province", "rate_area"]).size().rename("barangays").to_string())

Negros Occidental (excl. Bacolod): under 5 5.7534e-05, all ages 3.9700e-05 cases per person per day
Bacolod City: under 5 2.0274e-05, all ages 1.2100e-05 cases per person per day
Philippines: under 5 1.4356e-04, all ages 8.9100e-05 cases per person per day
province           rate_area                        
Bacolod City       Bacolod City                           7
Negros Occidental  Negros Occidental (excl. Bacolod)    163
Negros Oriental    Negros Occidental (excl. Bacolod)     46


In [8]:
impact = impact.merge(barangays[["psgc_code", "pop_total", "pop_under5", "pop_60plus", "y0_under5_per_day",
                                 "y0_all_per_day"]], on="psgc_code", how="left", validate="m:1")
show(impact, "barangay-days + population and rates")

for s in SCENARIOS_UGM3:
    delta_c = impact["p_exposed"] * impact["activity"] * s
    impact[f"delta_c_ugm3_s{s}"] = delta_c
    impact[f"pct_rise_s{s}"] = health.relative_increase(delta_c) * 100
    impact[f"extra_under5_s{s}"] = health.extra_cases(impact["pop_under5"], impact["y0_under5_per_day"], delta_c)
    impact[f"extra_60plus_s{s}"] = health.extra_cases(impact["pop_60plus"], impact["y0_all_per_day"], delta_c)
    impact[f"extra_all_s{s}"] = health.extra_cases(impact["pop_total"], impact["y0_all_per_day"], delta_c)
show(impact, "health impact")

barangay-days + population and rates: 221,184 rows x 14 columns
  missing: {'plume_index': 216}


health impact: 221,184 rows x 29 columns


  missing: {'plume_index': 216}


In [9]:
is_test = impact["split"] == "test"
scenario_summary = pd.DataFrame({
    s: {"mean_delta_c_ugm3": impact.loc[is_test, f"delta_c_ugm3_s{s}"].mean(),
        "max_delta_c_ugm3": impact.loc[is_test, f"delta_c_ugm3_s{s}"].max(),
        "max_pct_rise": impact.loc[is_test, f"pct_rise_s{s}"].max(),
        "extra_under5": impact.loc[is_test, f"extra_under5_s{s}"].sum(),
        "extra_60plus": impact.loc[is_test, f"extra_60plus_s{s}"].sum(),
        "extra_all": impact.loc[is_test, f"extra_all_s{s}"].sum()}
    for s in SCENARIOS_UGM3}).T.rename_axis("scenario_ugm3")
print(f"2026 totals ({impact.loc[is_test, 'date'].nunique()} days, {is_test.sum():,} barangay-days), "
      f"{MAIN_BASELINE} rates, RR {health.RR_PER_10_UGM3}")
scenario_summary.round(3)

2026 totals (294 days, 63,504 barangay-days), Negros Occidental and Bacolod City rates, RR 1.0126


,mean_delta_c_ugm3,max_delta_c_ugm3,max_pct_rise,extra_under5,extra_60plus,extra_all
scenario_ugm3,,,,,,
20,0.905,9.722,1.225,2.491,1.451,16.124
40,1.810,19.444,2.465,4.993,2.909,32.323
100,4.525,48.610,6.276,12.568,7.322,81.366


## Risk level and rank

The risk level uses delta_c under the main scenario: Low below 5, Moderate 5 to below 20, High 20 to below 40,
Very high 40 ug/m3 and above. The rank orders barangays within each date by extra_under5 + extra_60plus under
the main scenario (1 = most extra cases; ties share the lower rank).

In [10]:
DC_MAIN = f"delta_c_ugm3_s{MAIN_SCENARIO_UGM3}"
VULNERABLE_MAIN = [f"extra_under5_s{MAIN_SCENARIO_UGM3}", f"extra_60plus_s{MAIN_SCENARIO_UGM3}"]

impact["risk_level"] = health.risk_level(impact[DC_MAIN])
impact["extra_under5_60plus"] = impact[VULNERABLE_MAIN].sum(axis=1)
impact["risk_rank"] = (impact.groupby("date")["extra_under5_60plus"]
                       .rank(ascending=False, method="min").astype(int))
show(impact[["risk_level", "risk_rank"]], "risk level and rank")
print("Barangay-days per risk level, main scenario")
impact.groupby(["risk_level", "split"], observed=False).size().unstack("split")

risk level and rank: 221,184 rows x 2 columns
  missing: none
Barangay-days per risk level, main scenario


split,test,train
risk_level,,
Low,54998,133594
Moderate,8506,20488
High,0,3246
Very high,0,352


## Checks

On 2026 report days the SO2 increase and extra cases should be higher than on days without reports. We compare
means under the main scenario, and add the barangay-days that were actually reported.

In [11]:
reported = report_log.dropna(subset=["psgc_code"]).astype({"psgc_code": "int64"})
reported_keys = pd.MultiIndex.from_frame(reported[["psgc_code", "event_date"]])
test = impact[is_test].copy()
test["reported"] = pd.MultiIndex.from_frame(test[["psgc_code", "date"]]).isin(reported_keys)

check_groups = {
    "days without reports": test[test["report_day"] == 0],
    "report days": test[test["report_day"] == 1],
    "report days, reported barangays": test[test["reported"]],
}
report_check = pd.DataFrame({
    name: {"dates": g["date"].nunique(), "barangay_days": len(g), "mean_p_exposed": g["p_exposed"].mean(),
           "mean_activity": g["activity"].mean(), "mean_delta_c_ugm3": g[DC_MAIN].mean(),
           "mean_extra_under5_60plus": g["extra_under5_60plus"].mean()}
    for name, g in check_groups.items()}).T.astype({"dates": int, "barangay_days": int})
ratio = report_check.loc["report days"] / report_check.loc["days without reports"]
print(f"report days / days without reports: delta_c x{ratio['mean_delta_c_ugm3']:.2f}, "
      f"extra cases x{ratio['mean_extra_under5_60plus']:.2f}")
report_check.round(6)

report days / days without reports: delta_c x0.80, extra cases x0.81


,dates,barangay_days,mean_p_exposed,mean_activity,mean_delta_c_ugm3,mean_extra_under5_60plus
days without reports,289,62424,0.137142,0.328547,1.816135,0.000125
report days,5,1080,0.130146,0.292150,1.454372,0.000101
"report days, reported barangays",5,92,0.454141,0.267414,5.075098,0.000394


The check fails in 2026: the mean SO2 increase on report days is lower than on days without reports, because
both the forecast emission and the mean exposure probability were lower on those five days. On report days, the
barangays that were actually reported have a much higher increase than the average barangay, so the scores
separate places better than they separate days.

Every 2026 report day has ashfall reports and four of the five also have sulfur odor reports, so we split the
reported barangay-days instead of the days: ashfall only, sulfur odor only, or both. Means are over reported
barangay-days, so a day with more reports counts more.

In [12]:
hazards = (reported[reported["event_date"] > TRAIN_END]
           .assign(is_ash=lambda d: d["hazard"].str.contains("ashfall"),
                   is_odor=lambda d: d["hazard"].str.contains("sulfur odor"))
           .groupby(["psgc_code", "event_date"])[["is_ash", "is_odor"]].max()
           .reset_index()
           .rename(columns={"event_date": "date"}))
hazards["hazard_type"] = np.select([hazards["is_ash"] & hazards["is_odor"], hazards["is_ash"]],
                                   ["both", "ashfall only"], default="sulfur odor only")
reported_2026 = test.merge(hazards[["psgc_code", "date", "hazard_type"]], on=["psgc_code", "date"],
                           how="inner", validate="1:1")
show(reported_2026, "2026 reported barangay-days with hazard")
assert len(reported_2026) == test["reported"].sum()
print(pd.crosstab(reported_2026["date"].dt.date, reported_2026["hazard_type"]).to_string())

hazard_check = reported_2026.groupby("hazard_type").agg(
    dates=("date", "nunique"), barangay_days=("psgc_code", "size"),
    mean_so2_forecast_td=("so2_forecast_td", "mean"), mean_p_exposed=("p_exposed", "mean"))
for row in hazard_check.itertuples():
    print(f"{row.Index}: {row.barangay_days} barangay-days on {row.dates} dates, "
          f"mean forecast SO2 {row.mean_so2_forecast_td:,.0f} t/d, mean p_exposed {row.mean_p_exposed:.3f}")

2026 reported barangay-days with hazard: 92 rows x 34 columns


  missing: none
hazard_type  ashfall only  both  sulfur odor only
date                                             
2026-02-19             18     9                 0
2026-02-26             12     1                 9
2026-03-15             14     0                 2
2026-07-09             10     6                 0
2026-10-10              0     8                 3
ashfall only: 54 barangay-days on 4 dates, mean forecast SO2 1,262 t/d, mean p_exposed 0.443
both: 24 barangay-days on 4 dates, mean forecast SO2 1,571 t/d, mean p_exposed 0.493
sulfur odor only: 14 barangay-days on 3 dates, mean forecast SO2 1,242 t/d, mean p_exposed 0.432


Forecast SO2 and p_exposed are similar across the three groups, slightly higher where both hazards were
reported. The groups are small (14 to 54 barangay-days), so we do not read more into the differences.

Barangay-days per risk level in 2026 under the main scenario, on quiet days (no reports) and report days.

In [13]:
test["day_type"] = np.where(test["report_day"] == 1, "report days", "quiet days")
risk_counts = (test.groupby(["risk_level", "day_type"], observed=False).size()
               .unstack("day_type")[["quiet days", "report days"]])
risk_counts.join((risk_counts / risk_counts.sum()).add_suffix(" (share)").round(3))

day_type,quiet days,report days,quiet days (share),report days (share)
risk_level,,,,
Low,54023,975,0.865,0.903
Moderate,8401,105,0.135,0.097
High,0,0,0.000,0.000
Very high,0,0,0.000,0.000


No 2026 barangay-day reaches High or Very high under the main scenario: with activity below 0.5, delta_c stays
below 20 ug/m3 even where p_exposed is close to 1. Report days have a smaller share of Moderate barangay-days than
quiet days, which matches the failed check above.

## Sensitivity

We compute the 2026 total of extra_under5 + extra_60plus for 18 settings: RR 1.001, 1.0126 or 1.020; the main rates
(Negros Occidental, with Bacolod City's own rates for its barangays) or the Philippine rates for every barangay
(0.0000891 for all ages, 0.0524 / 365 for under 5); and S = 20, 40 or 100 ug/m3. For each setting we rank barangays
by their 2026 total and compare the ranking with the main setting (RR 1.0126, main rates, 40 ug/m3) by Spearman
correlation.

In [14]:
BASELINE_SETTINGS = {
    MAIN_BASELINE: (test["y0_under5_per_day"].to_numpy(), test["y0_all_per_day"].to_numpy()),
    PHILIPPINES_AREA: (DAILY_RATES[PHILIPPINES_AREA]["under5"], DAILY_RATES[PHILIPPINES_AREA]["all_ages"]),
}
exposure_per_ugm3 = test["p_exposed"] * test["activity"]
sensitivity_rows, barangay_totals = [], {}
for rr, (baseline, (y0_under5, y0_all)), s in itertools.product(RR_SENSITIVITY, BASELINE_SETTINGS.items(),
                                                                SCENARIOS_UGM3):
    delta_c = exposure_per_ugm3 * s
    under5 = pd.Series(health.extra_cases(test["pop_under5"], y0_under5, delta_c, rr), index=test.index)
    aged60plus = pd.Series(health.extra_cases(test["pop_60plus"], y0_all, delta_c, rr), index=test.index)
    barangay_totals[(rr, baseline, s)] = (under5 + aged60plus).groupby(test["psgc_code"]).sum()
    sensitivity_rows.append({"rr_per_10ugm3": rr, "baseline_rates": baseline, "scenario_ugm3": s,
                             "extra_under5": under5.sum(), "extra_60plus": aged60plus.sum(),
                             "extra_under5_60plus": under5.sum() + aged60plus.sum()})

main_key = (health.RR_PER_10_UGM3, MAIN_BASELINE, MAIN_SCENARIO_UGM3)
main_totals = barangay_totals[main_key]
assert np.isclose(main_totals.sum(), test["extra_under5_60plus"].sum())

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity["main_setting"] = [key == main_key for key in barangay_totals]
sensitivity["ratio_to_main"] = sensitivity["extra_under5_60plus"] / main_totals.sum()
sensitivity["rank_spearman_vs_main"] = [totals.corr(main_totals, method="spearman")
                                        for totals in barangay_totals.values()]
show(sensitivity, "health_sensitivity")
print(f"2026 extra cases, under 5 + 60 and over: main {main_totals.sum():.2f}, range "
      f"{sensitivity['extra_under5_60plus'].min():.2f} to {sensitivity['extra_under5_60plus'].max():.2f}")
print(f"lowest Spearman correlation with the main ranking: {sensitivity['rank_spearman_vs_main'].min():.4f}")
sensitivity.round(4)

health_sensitivity: 18 rows x 9 columns
  missing: none
2026 extra cases, under 5 + 60 and over: main 7.90, range 0.31 to 89.48
lowest Spearman correlation with the main ranking: 0.9971


,rr_per_10ugm3,baseline_rates,scenario_ugm3,extra_under5,extra_60plus,extra_under5_60plus,main_setting,ratio_to_main,rank_spearman_vs_main
0,1.0010,Negros Occidental and Bacolod City,20,0.1984,0.1156,0.3140,False,0.0397,1.0000
1,1.0010,Negros Occidental and Bacolod City,40,0.3969,0.2312,0.6281,False,0.0795,1.0000
2,1.0010,Negros Occidental and Bacolod City,100,0.9927,0.5784,1.5711,False,0.1988,1.0000
3,1.0010,Philippines,20,0.5754,0.3123,0.8877,False,0.1123,0.9971
4,1.0010,Philippines,40,1.1509,0.6248,1.7757,False,0.2247,0.9971
5,1.0010,Philippines,100,2.8788,1.5628,4.4416,False,0.5621,0.9971
6,1.0126,Negros Occidental and Bacolod City,20,2.4906,1.4512,3.9418,False,0.4989,1.0000
7,1.0126,Negros Occidental and Bacolod City,40,4.9927,2.9090,7.9016,True,1.0000,1.0000
8,1.0126,Negros Occidental and Bacolod City,100,12.5683,7.3225,19.8908,False,2.5173,1.0000
9,1.0126,Philippines,20,7.2225,3.9207,11.1433,False,1.4102,0.9971


The 2026 total spans more than two orders of magnitude across settings, mostly through RR and S, which enter almost
linearly, but the ranking of barangays barely moves. Under the main rates, RR and S only change the curvature of
RR^(delta_c/10) - 1, so the Spearman correlation stays at 1.000; the Philippine setting gives the 7 Bacolod City
barangays the same rates as every other barangay, which moves them up and lowers the correlation slightly.

## Top barangays in 2026

The 10 barangays with the largest 2026 total of extra_under5 + extra_60plus under the main setting. Municipality
names are written as in the census.

In [15]:
barangay_2026 = (test.groupby("psgc_code")
                 .agg(mean_delta_c_ugm3=(DC_MAIN, "mean"),
                      days_moderate_or_higher=("risk_level", lambda s: int((s != "Low").sum())),
                      extra_under5=(VULNERABLE_MAIN[0], "sum"),
                      extra_60plus=(VULNERABLE_MAIN[1], "sum"),
                      extra_under5_60plus=("extra_under5_60plus", "sum"))
                 .reset_index()
                 .merge(barangays[["psgc_code", "barangay", "municipality", "province", "dist_km",
                                   "pop_under5", "pop_60plus"]], on="psgc_code", how="left", validate="1:1"))
show(barangay_2026, "barangay totals, 2026")

top_barangays = (barangay_2026.sort_values("extra_under5_60plus", ascending=False).head(10)
                 .reset_index(drop=True))
top_barangays.insert(0, "rank", np.arange(1, len(top_barangays) + 1))
top_barangays = top_barangays[["rank", "psgc_code", "barangay", "municipality", "province", "dist_km", "pop_under5",
                               "pop_60plus", "mean_delta_c_ugm3", "days_moderate_or_higher", "extra_under5",
                               "extra_60plus", "extra_under5_60plus"]]
show(top_barangays, "top_barangays_2026")
print(f"share of the 2026 total in the top 10: "
      f"{top_barangays['extra_under5_60plus'].sum() / barangay_2026['extra_under5_60plus'].sum():.1%}")
top_barangays.round(3)

barangay totals, 2026: 216 rows x 12 columns
  missing: none
top_barangays_2026: 10 rows x 13 columns
  missing: none
share of the 2026 total in the top 10: 35.4%


,rank,psgc_code,barangay,municipality,province,dist_km,pop_under5,pop_60plus,mean_delta_c_ugm3,days_moderate_or_higher,extra_under5,extra_60plus,extra_under5_60plus
0,1,604517013,Sag-Ang,LA CASTELLANA,Negros Occidental,9.232,1633,1051,8.337,224,0.290,0.129,0.419
1,2,604524004,Codcod,CITY OF SAN CARLOS,Negros Occidental,10.854,1646,1102,7.754,220,0.272,0.126,0.398
2,3,604502014,Ma-ao Barrio,CITY OF BAGO,Negros Occidental,16.841,1728,1756,5.913,163,0.218,0.153,0.370
3,4,604517012,Robles,LA CASTELLANA,Negros Occidental,15.196,1966,2092,4.614,125,0.193,0.142,0.335
4,5,604502016,Mailum,CITY OF BAGO,Negros Occidental,7.493,1014,852,9.312,249,0.201,0.117,0.318
5,6,604520019,Minoyan,MURCIA,Negros Occidental,11.666,1540,877,5.246,149,0.172,0.068,0.239
6,7,630200051,Mansilingan,CITY OF BACOLOD (Capital),Bacolod City,28.235,4420,5743,3.822,89,0.127,0.098,0.225
7,8,604502004,Bacong-Montilla,CITY OF BAGO,Negros Occidental,15.376,1025,771,6.086,167,0.133,0.069,0.202
8,9,604502007,Binubuhan,CITY OF BAGO,Negros Occidental,14.683,585,641,7.044,190,0.088,0.066,0.154
9,10,604502022,Tabunan,CITY OF BAGO,Negros Occidental,28.146,881,689,4.867,135,0.091,0.049,0.140


Eight of the ten are within 17 km of the crater, with higher SO2 increases and more Moderate days. The two
exceptions, both about 28 km away, are Mansilingan, the largest Bacolod City barangay in the study area, and
Tabunan in Bago City.

## Monthly totals

Total extra_under5 + extra_60plus per month under the main setting, summed over the 216 barangays; 2024-2025 values
use in-sample forecasts. January 2024 starts on the 2nd (no forecast for the 1st) and October 2026 ends on the 21st.

In [16]:
daily_totals = (impact.assign(high_or_very_high=impact["risk_level"].isin(["High", "Very high"]))
                .groupby("date")
                .agg(split=("split", "first"), report_day=("report_day", "first"),
                     so2_forecast_td=("so2_forecast_td", "first"),
                     extra_under5=(VULNERABLE_MAIN[0], "sum"), extra_60plus=(VULNERABLE_MAIN[1], "sum"),
                     extra_under5_60plus=("extra_under5_60plus", "sum"),
                     barangays_high_or_very_high=("high_or_very_high", "sum"))
                .reset_index())
show(daily_totals, "daily totals")

health_by_month = (daily_totals.groupby(daily_totals["date"].dt.to_period("M").rename("month"))
                   .agg(split=("split", "first"), days=("date", "size"), report_days=("report_day", "sum"),
                        extra_under5=("extra_under5", "sum"), extra_60plus=("extra_60plus", "sum"),
                        extra_under5_60plus=("extra_under5_60plus", "sum"))
                   .reset_index())
health_by_month["month"] = health_by_month["month"].astype(str)
show(health_by_month, "health_by_month")
health_by_month.round(3)

daily totals: 1,024 rows x 8 columns
  missing: none
health_by_month: 34 rows x 7 columns
  missing: none


,month,split,days,report_days,extra_under5,extra_60plus,extra_under5_60plus
0,2024-01,train,30,0,0.248,0.150,0.398
1,2024-02,train,29,0,0.210,0.132,0.342
2,2024-03,train,31,0,0.326,0.196,0.522
3,2024-04,train,30,0,0.297,0.187,0.484
4,2024-05,train,31,0,0.336,0.202,0.538
5,2024-06,train,30,1,0.516,0.312,0.827
6,2024-07,train,31,0,0.645,0.377,1.022
7,2024-08,train,31,0,0.839,0.489,1.328
8,2024-09,train,30,2,1.797,1.027,2.824
9,2024-10,train,31,1,1.277,0.740,2.017


Monthly totals peak from September 2024 to January 2025, at 2.0 to 2.8 extra cases per month. In 2026 they range
from about 0.5 to 1.1 per month.

## Top risk days

The 10 days with the largest total extra_under5 + extra_60plus across the study barangays, 2024 to 2026, under
the main setting, with the number of barangays at High or Very high that day.

In [17]:
top_risk_days = (daily_totals.sort_values("extra_under5_60plus", ascending=False).head(10)
                 .reset_index(drop=True))
top_risk_days.insert(0, "rank", np.arange(1, len(top_risk_days) + 1))
top_risk_days["date"] = top_risk_days["date"].dt.date
top_risk_days = top_risk_days[["rank", "date", "split", "report_day", "so2_forecast_td", "extra_under5_60plus",
                               "barangays_high_or_very_high"]]
show(top_risk_days, "top_risk_days")
top_day_2026 = daily_totals.loc[daily_totals.loc[daily_totals["split"] == "test", "extra_under5_60plus"].idxmax()]
print(f"highest 2026 day: {top_day_2026['date'].date()}, {top_day_2026['extra_under5_60plus']:.3f} extra cases, "
      f"forecast SO2 {top_day_2026['so2_forecast_td']:,.0f} t/d")
top_risk_days.round(3)

top_risk_days: 10 rows x 7 columns
  missing: none
highest 2026 day: 2026-05-08, 0.065 extra cases, forecast SO2 2,146 t/d


,rank,date,split,report_day,so2_forecast_td,extra_under5_60plus,barangays_high_or_very_high
0,1,2024-09-25,train,0,6394.3,0.213,79
1,2,2024-09-23,train,0,6569.5,0.198,67
2,3,2024-09-24,train,1,5982.9,0.196,69
3,4,2024-09-22,train,0,6505.2,0.175,48
4,5,2024-09-20,train,0,6782.9,0.171,53
5,6,2024-09-30,train,0,5979.7,0.165,52
6,7,2024-11-26,train,0,6501.8,0.161,50
7,8,2024-11-30,train,0,5692.8,0.156,51
8,9,2024-11-02,train,0,6172.2,0.154,50
9,10,2024-10-02,train,0,5413.4,0.149,52


All 10 days fall between September and November 2024 (in-sample forecasts), with forecast emission above 5,000 t/d,
and only one of them (2024-09-24) is a report day. The highest 2026 day has less than half the total of the tenth
day on this list.

## Save

`health_impact.csv` has one row per barangay-day with a forecast. Columns ending in `_s20`, `_s40` and `_s100`
belong to each scenario; `risk_level` and `risk_rank` use the main scenario.

In [18]:
SCENARIO_COLUMNS = ["delta_c_ugm3", "pct_rise", "extra_under5", "extra_60plus", "extra_all"]
HEALTH_COLUMNS = (["psgc_code", "date", "split", "report_day", "p_exposed", "so2_forecast_td", "activity"]
                  + [f"{col}_s{s}" for s in SCENARIOS_UGM3 for col in SCENARIO_COLUMNS]
                  + ["risk_level", "risk_rank"])
health_impact = impact[HEALTH_COLUMNS].copy()
health_impact["date"] = health_impact["date"].dt.date
show(health_impact, "health_impact")
health_impact.to_csv(PROCESSED / "health_impact.csv", index=False, float_format="%.6g")
sensitivity.to_csv(TABLES / "health_sensitivity.csv", index=False, float_format="%.4f")
top_barangays.to_csv(TABLES / "top_barangays_2026.csv", index=False, float_format="%.4f")
health_by_month.to_csv(TABLES / "health_by_month.csv", index=False, float_format="%.4f")
top_risk_days.to_csv(TABLES / "top_risk_days.csv", index=False, float_format="%.4f")
print(f"health_impact.csv: {(PROCESSED / 'health_impact.csv').stat().st_size / 1e6:.0f} MB")

health_impact: 221,184 rows x 24 columns
  missing: none


health_impact.csv: 50 MB


In [19]:
metrics["health_impact"] = {
    "q_ref_td": float(Q_REF_TD),
    "q_ref_n_days": len(q_ref_days),
    "activity_cap": ACTIVITY_CAP,
    "rr_per_10ugm3": health.RR_PER_10_UGM3,
    "scenarios_ugm3": SCENARIOS_UGM3,
    "main_scenario_ugm3": MAIN_SCENARIO_UGM3,
    "main_baseline": MAIN_BASELINE,
    "activity_2026": activity_summary.loc["test", ["min", "median", "max", "days_at_cap"]].astype(float).round(4).to_dict(),
    "totals_2026": scenario_summary[["extra_under5", "extra_60plus", "extra_all"]].round(4)
                   .rename(index=str).to_dict(orient="index"),
    "report_day_check_2026": report_check.round(6).to_dict(orient="index"),
    "hazard_check_2026": hazard_check.round(4).to_dict(orient="index"),
    "risk_level_counts_2026": risk_counts.astype(int).rename(index=str).to_dict(orient="index"),
    "sensitivity_total_range": [round(float(sensitivity["extra_under5_60plus"].min()), 4),
                                round(float(sensitivity["extra_under5_60plus"].max()), 4)],
    "min_rank_spearman_vs_main": round(float(sensitivity["rank_spearman_vs_main"].min()), 4),
}
metrics_path = RESULTS / "metrics.json"
all_metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
all_metrics.update(metrics)
metrics_path.write_text(json.dumps(all_metrics, indent=2, default=float))
print("metrics keys:", list(all_metrics))

metrics keys: ['so2_next_day', 'drift_next_day', 'plume_index', 'exposure', 'health_impact', 'allocation']


## Interpretation

These numbers are expected values, not counted cases: a published exposure-response value (RR = 1.0126 per 10 ug/m3,
Carlsen et al., 2021) applied to 2020 census populations, FHSIS 2022 baseline rates and assumed SO2 scenarios, not
measured concentrations. The FHSIS 2022 baseline is likely a lower bound, since it counts only cases reported
through public health facilities; the Philippine rate is more than twice the Negros Occidental rate.